# Validación de AutoML P2 en Google Colab
Selecciona GPU en el entorno de ejecución. Sube el ZIP del proyecto. Los CSV se leen desde una carpeta de Drive indicada durante la ejecución; no es necesario editar las celdas. La configuración por defecto ejecuta HPO, benchmark y backtesting completo. Datos: p1–p6 desarrollo, p7–p10 evaluación temporal.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import zipfile, os, subprocess, sys
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    z.extractall('/content/automl_validacion')
project = next(Path('/content/automl_validacion').rglob('main.py')).parent
os.chdir(project)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
drive.mount('/content/drive')
input_dir = input('Ruta de Drive que contiene los 10 CSV: ').strip()
assert all((Path(input_dir)/f'p{i}_extrac.csv').is_file() for i in range(1,11)), 'Faltan CSV'


In [ ]:
subprocess.run([sys.executable,'main.py','--input',input_dir,'--config','config/full.yaml','--parallel'],check=True)

In [ ]:
import json
from IPython.display import display, Image
runs = [p for p in Path('data/output').glob('*/results_summary.json')
        if json.loads(p.read_text(encoding='utf-8')).get('stage')=='completed_backtesting']
run = max(runs,key=lambda p:p.stat().st_mtime).parent
summary = json.loads((run/'results_summary.json').read_text(encoding='utf-8'))
assert len(summary['models'])==9
assert summary['parallelism']['speed_up']>0
for name in summary['models']:
    meta=json.loads((run/f'{name}_metadata.json').read_text(encoding='utf-8'))
    assert {'hpo','parallelism','backtesting'}<=meta.keys()
    assert len(meta['backtesting']['waves'])==4
    assert (run/f'{name}_model.pkl').exists()
print('Campeón:',summary['campeon'])
print('Razón:',summary['selection_reason'])
print('Speed-up:',summary['parallelism']['speed_up'])
display(Image(filename=str(run/'auc_por_ola.png')))